# 1. The roadmap as an asymmetric TSP / SOP, and its bounds
**Class:** Session 6 slide 3 (ATSP formulation, assignment relaxation), Session 9 slide 22 (load variables Q<sub>i</sub>), Session 5 slides 16–28 (LP bound, branch and bound), Session 9 slides 23–25 (valid inequalities).

**IIMA context:** a student who wants WACC (Corporate Finance) and variance analysis (Management Accounting) asks for the fastest prerequisite-respecting order. With ~13 topics, B&B proves the optimum in well under a second.

In [1]:
import os, sys
ROOT = os.path.abspath('..'); sys.path.insert(0, os.path.join(ROOT, 'src'))
from speedrun.data import load_web, Params, Profile
from speedrun.graph import build_instance
from speedrun.costs import Evaluator
web = load_web(os.path.join(ROOT, 'data', 'sample'))
P = Params.load(os.path.join(ROOT, 'data', 'sample', 'params.json'))
inst = build_instance(web, Profile(), P); ev = Evaluator(inst)
name = lambda ids: ' -> '.join(ids)
print(ev.n, 'topics,', round(ev.sum_p, 1), 'study hours (estimates)')

28 topics, 71.5 study hours (estimates)


In [2]:
from speedrun import milp, heuristics as H
ev13 = Evaluator(build_instance(web, Profile(targets=['F8', 'M6']), P))
print(ev13.n, 'topics in scope:', ev13.ids)
seq, zh = H.solve_heuristic(ev13, restarts=2)
r = milp.solve_exact(ev13, time_limit=60, warm_seq=seq)
print('heuristic z^H =', round(zh, 3), '| exact z* =', round(r['objective_eval'], 3), '| proven:', r['optimal'], '| nodes:', r['nodes'])

9 topics in scope: ['Q1', 'M1', 'M5', 'M6', 'F1', 'F2', 'F6', 'F7', 'F8']


heuristic z^H = 22.488 | exact z* = 22.488 | proven: True | nodes: 0


The chain of lower bounds (minimisation, so Z<sub>LP</sub> ≤ Z<sub>IP</sub>, Session 5 slide 16). Relaxing constraint (III) leaves an assignment problem (Session 6 slide 3).

In [3]:
print('assignment relaxation (+ arc fixing):', round(milp.assignment_bound(ev13), 3))
print('LP relaxation, no cuts             :', round(milp.lp_bound(ev13, cuts=False), 3))
print('LP + arc fixing + 2-cycle cuts     :', round(milp.lp_bound(ev13, cuts=True), 3))
print('optimum z*                         :', round(r['objective_eval'], 3))
print(name(ev13.to_ids(r['seq'])))

assignment relaxation (+ arc fixing): 22.488
LP relaxation, no cuts             : 21.81
LP + arc fixing + 2-cycle cuts     : 22.488
optimum z*                         : 22.488
Q1 -> F6 -> F7 -> F1 -> F2 -> F8 -> M1 -> M5 -> M6


**Reading:** precedence-based arc fixing (x<sub>jk</sub> = 0 when k is an ancestor of j) does here what subtour and capacity inequalities do on Session 9 slide 25: it closes most of the LP gap.